<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# SNO Public Solar-Neutrino Data — Online Data EDA

This notebook reads the provider's public online resources directly, without using the converted TPeanuts tables. It inventories the remote content, checks formats and units, and determines which canonical products can be generated.

Official source: [https://sno.phy.queensu.ca/sno/prlwebpage/](https://sno.phy.queensu.ca/sno/prlwebpage/)

## Available products

- Detector observations: day/night reconstructed-energy counts, backgrounds and zenith exposure.
- Flux: experimental CC/NC/ES results are publication-level quantities.
- Probability: model-dependent fitted results, not a production table in this release.
- Solar density, radial production and production spectra: unavailable.

## 1. Imports and remote access

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Online inventory and format inspection

In [2]:
BASE = "https://sno.phy.queensu.ca/sno/prlwebpage"
URLS = {"day_night": f"{BASE}/SnoDayNightPrlSpectra.dat", "coszenith": f"{BASE}/SnoCosZenith.dat", "background": f"{BASE}/SnoBackgroundTables.dat", "howto": f"{BASE}/prldata.ps"}
rows, remote = [], {}
for name, url in URLS.items():
    payload = fetch(url)
    remote[name] = payload
    rows.append({"resource": name, "bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest(), "url": url})
display(pd.DataFrame(rows))
print(remote["day_night"].decode(errors="replace")[:1200])

,resource,bytes,sha256,url
0,day_night,809,fef0910f755a68abb6877af0c9f9f74f34b448364125fc...,https://sno.phy.queensu.ca/sno/prlwebpage/SnoD...
1,coszenith,7893,894307855b6eb5854c774e08bcffe5555e909bfd52d512...,https://sno.phy.queensu.ca/sno/prlwebpage/SnoC...
2,background,1914,53d53cf9b19d0c2699cfd725441b4700bcd59c535e90f5...,https://sno.phy.queensu.ca/sno/prlwebpage/SnoB...
3,howto,209704,c6abb4130af2be2afda3c31b24aff6dce766cfe883f8d6...,https://sno.phy.queensu.ca/sno/prlwebpage/prld...


# SnoDayNightPrlSpectra.dat
# 
# Data shown in figure 2 of
# nucl-ex/0204009
#
# livetime day   =  128.5 days
#          night =  177.9 days
# 
# bin  Tmin   Tmax    Nday   Nnight
#      (MeV)  (MeV) 
#
   1    5.0    5.5     191    301
   2    5.5    6.0     180    236
   3    6.0    6.5     163    205
   4    6.5    7.0     121    188
   5    7.0    7.5     104    177
   6    7.5    8.0      81    133
   7    8.0    8.5      70     92
   8    8.5    9.0      76    101
   9    9.0    9.5      49     72
  10    9.5   10.0      45     65
  11   10.0   10.5      36     47
  12   10.5   11.0      27     45
  13   11.0   11.5      17     31
  14   11.5   12.0      10     16
  15   12.0   12.5       5     14
  16   12.5   13.0       6     12
  17   13.0   20.0       5      7



## 3. Conclusions

Only source-published observables are attributed to this provider. Derived TPeanuts quantities and detector-level spectra are labelled separately by the generator notebook.